# Notebook 05 — Earth Engine Exposure Extraction
**GeoAI for Climate, Environment and Health** | 25 June 2026

---

## Learning objectives
- Know which satellite/reanalysis products map to which environmental variables
- See a short worked extraction from Google Earth Engine (optional — skip if not set up)
- Understand the Earth Engine Code Editor workflow for point sampling
- Load and explore the pre-generated synthetic exposure grid

**Time:** ~20 minutes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys, pathlib

def _find_workshop_dir():
    here = pathlib.Path().resolve()
    for candidate in (here, here.parent, here.parent.parent):
        if (candidate / 'data').exists():
            return candidate
    return here

WORKSHOP_DIR = _find_workshop_dir()  # works whether run flat or from a notebooks/ subfolder
DATA_DIR = WORKSHOP_DIR / 'data'
sys.path.insert(0, str(WORKSHOP_DIR))
from shared_utils import find_nearest_grid_cells

villages = pd.read_csv(DATA_DIR / 'villages.csv')
print('Ready.')

---
## 📖 1. Where does environmental data come from?

| Variable | Source | Temporal resolution | Notes |
|---|---|---|---|
| Temperature (mean/max) | **ERA5-Land** (ECMWF) | Hourly → daily aggregate | Reanalysis: combines model + observations |
| PM2.5 / air quality | **CAMS** (Copernicus) | 3-hourly → daily | Global atmospheric composition |
| NDVI (vegetation) | **MODIS** (Terra/Aqua) | **16-day composites** | Real data is NOT daily — see caveat below |
| Precipitation | **CHIRPS** | Daily | Satellite IR + gauge stations |
| Relative humidity | Derived (Magnus-Tetens) | Same as temp source | No direct satellite measurement |

> ⚠️ **NDVI temporal resolution caveat:** Real MODIS NDVI products (e.g. MOD13A3) are 16-day composites — the sensor revisits each location every 16 days and takes the best-quality pixel in that window. Our synthetic dataset uses daily values to simplify the teaching, but in real analyses you must account for this: either aggregate your exposure windows to ≥16 days or use the 16-day value for the entire window.

All of these products are freely available via **Google Earth Engine** and can be extracted for any set of point locations or polygons.

---
## ✅ Section 2 — In-notebook Earth Engine demo (optional)

**If Earth Engine is not configured on your account, skip to Section 3 — nothing later depends on this section.**

The cell below tries to authenticate and run a small extraction. If it fails, it prints a skip message and continues.

In [ ]:
# ── OPTIONAL: Earth Engine extraction demo ──────────────────────────────────
# This entire section is wrapped in try/except. If EE is unavailable, skip it.

EE_AVAILABLE = False
try:
    import ee

    # Try service-account auth first (server environment),
    # then fall back to interactive auth.
    SA_KEY = WORKSHOP_DIR.parent / 'palsearth' / 'palsearth-sa-key.json'
    try:
        credentials = ee.ServiceAccountCredentials(
            email='ee-service-account@your-project.iam.gserviceaccount.com',
            key_file=str(SA_KEY)
        )
        ee.Initialize(credentials)
    except Exception:
        ee.Authenticate()
        ee.Initialize()

    EE_AVAILABLE = True
    print('✅ Earth Engine initialised')

except Exception as ee_err:
    print(f'⏭  Earth Engine not available: {ee_err}')
    print('   Skipping EE demo cells — the rest of the notebook runs without them.')

In [ ]:
if EE_AVAILABLE:
    # Sample ERA5-Land daily mean temperature at the 10 village locations
    # for a single date (2024-07-15) to show the extraction workflow.

    village_fc = ee.FeatureCollection([
        ee.Feature(ee.Geometry.Point([row['longitude'], row['latitude']]),
                   {'village': row['village']})
        for _, row in villages.iterrows()
    ])

    era5 = (
        ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
        .filterDate('2024-07-15', '2024-07-16')
        .select(['temperature_2m_max', 'temperature_2m_min'])
        .first()
    )

    # Convert Kelvin to Celsius.
    era5_c = era5.subtract(273.15).rename(['temp_max_c', 'temp_min_c'])

    sampled = era5_c.sampleRegions(
        collection=village_fc,
        scale=11132,   # ERA5-Land native resolution ~11 km
        geometries=False
    )

    ee_df = pd.DataFrame(sampled.getInfo()['features'])
    ee_df = pd.json_normalize(ee_df['properties'].tolist())
    print('ERA5-Land extraction (real satellite data):')
    print(ee_df[['village', 'temp_max_c', 'temp_min_c']].to_string(index=False))
    print()
    print('⚠️  These values use real ERA5 data — they will not match our synthetic grid exactly,'
          ' but the *shape of the workflow* is identical.')
else:
    print('EE demo skipped.')

---
## 📖 3. Earth Engine Code Editor walkthrough

For larger extractions — many locations, many dates — it is more practical to run the extraction as a batch job in the **Earth Engine Code Editor** (code.earthengine.google.com) and export results to Google Drive.

The script `../ee_scripts/sample_points_demo.js` does the following. Walk through each block:

```javascript
// 1. Define the 10 village point locations as a FeatureCollection.
//    (In a real project, upload your CSV via Assets → New → Table upload)
var villages = ee.FeatureCollection([...]);

// 2. Load the ERA5-Land daily image collection, filter to date range.
var era5 = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
  .filterDate('2023-01-01', '2024-07-31')
  .select(['temperature_2m_max']);

// 3. For each image in the collection, sample at all village points.
var sampled = era5.map(function(img) {
  return img.reduceRegions(villages, ee.Reducer.first(), 11132)
            .map(function(f) {
              return f.set('date', img.date().format('YYYY-MM-dd'));
            });
}).flatten();

// 4. Print a preview in the Console panel.
print('Sample rows:', sampled.limit(5));

// 5. Export to Google Drive (uncomment to run as a Task).
// Export.table.toDrive({collection: sampled, description: 'village_era5_temps', fileFormat: 'CSV'});
```

**Where to click:**
- **Assets panel** (top-left): upload a CSV of locations as a *Table asset*, then replace the hardcoded `villages` FeatureCollection with `ee.FeatureCollection('users/your_username/villages')`.
- **Run button** (top of code editor): executes the script; results appear in the **Console** tab.
- **Tasks panel** (top-right): after uncommenting the `Export` call, click **RUN** next to your task to start the export to Drive.

> *Facilitators: add screenshots of the Assets panel and Tasks panel as `../images/ee_asset_upload.png` and `../images/ee_tasks.png` before the event.*

---
## ✅ Worked example — Load and explore the exposure grid

In [ ]:
# The pre-generated grid covers ~700 days across 30×27 grid cells.
# We read just one date to get a spatial snapshot.
exposure_grid = pd.read_csv(DATA_DIR / 'exposure_grid.csv', parse_dates=['date'])
print(f'Full grid shape: {exposure_grid.shape}')
print(f'Date range: {exposure_grid["date"].min().date()} → {exposure_grid["date"].max().date()}')
print(f'Unique grid cells: {exposure_grid["grid_id"].nunique()}')

snapshot = exposure_grid[exposure_grid['date'] == '2024-07-15']
print(f'\nSnapshot (2024-07-15) rows: {len(snapshot)}')

# 2×2 spatial map of the snapshot.
vars_info = [
    ('temp_mean_c',      'Temperature (°C)',  'RdYlBu_r'),
    ('pm25_ugm3',        'PM2.5 (µg/m³)',     'YlOrRd'),
    ('ndvi',             'NDVI',              'YlGn'),
    ('precipitation_mm', 'Precipitation (mm)','Blues'),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, (col, title, cmap) in zip(axes.ravel(), vars_info):
    sc = ax.scatter(snapshot['lon'], snapshot['lat'], c=snapshot[col], cmap=cmap, s=14, alpha=0.85)
    plt.colorbar(sc, ax=ax, shrink=0.75)
    ax.set_title(title)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
fig.suptitle('Exposure grid snapshot — 2024-07-15', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

---
## 🧪 Exercise — Seasonal temperature pattern at a village  *(3 points)*

1. Pick any village from `villages.csv`.
2. Find the nearest grid cell to that village using `find_nearest_grid_cells()`.
3. Filter `exposure_grid` to that grid cell and plot `temp_mean_c` over time (x = date, y = temperature).
4. Identify the **month** when temperature peaks.

Store the village name in `chosen_village` (a string) and the peak month number (1–12) in `peak_month` (an int).

In [ ]:
chosen_village = villages['village'].iloc[0]  # change to any village name you like
v_row = villages[villages['village'] == chosen_village]
gid = find_nearest_grid_cells(v_row, exposure_grid.drop_duplicates('grid_id'))['grid_id'].iloc[0]

# Hint: filter exposure_grid to rows where grid_id == gid, plot temp_mean_c
# against date with plt.plot(), then read off the month with the highest
# temperature and store its number (1-12) in peak_month.

cell_ts = exposure_grid[exposure_grid['grid_id'] == gid].copy()
cell_ts['month'] = cell_ts['date'].dt.month

plt.figure(figsize=(11, 4))
plt.plot(cell_ts['date'], cell_ts['temp_mean_c'], color='#e63946', alpha=0.6, linewidth=0.8)
plt.xlabel('Date')
plt.ylabel('Mean temperature (°C)')
plt.title(f'Daily temperature at nearest grid cell to {chosen_village}')
plt.tight_layout()
plt.show()

monthly_mean = cell_ts.groupby('month')['temp_mean_c'].mean()
peak_month = int(monthly_mean.idxmax())

print(f'Chosen village: {chosen_village}  |  Grid cell: {gid}')
print(f'Peak temperature month: {peak_month} ({pd.Timestamp(2024, peak_month, 1).strftime("%B")})')


---
### ✅ Knowledge Check — Seasonality of environmental exposure

*No coding here — just check your understanding before moving on.*

**Q1. Why does it matter, for a study of pregnancy and heat exposure, to know *which month* temperatures peak?**

- **A.** It doesn't matter — temperature is constant year-round everywhere
- **B.** If a participant's pregnancy trimester overlaps with the hottest months, her heat exposure during that critical window will be higher than for someone pregnant during cooler months
- **C.** Peak month only matters for agricultural planning
- **D.** Monthly patterns are only relevant for malaria studies

<details>
<summary>Show answer</summary>

**Answer: B** — Knowing the seasonal pattern lets you identify whether a participant's pregnancy (and especially sensitive windows like the third trimester) coincided with the period of highest heat exposure.

</details>

**Q2. Earth Engine / satellite-derived datasets like the one used here typically provide environmental data as a grid of cells rather than a value at every exact point. What does this mean for village-level analysis?**

- **A.** Each village automatically gets its own perfectly matched cell
- **B.** Villages are assigned the value of their nearest grid cell, which is an approximation of their true local exposure
- **C.** Gridded data cannot be used for village-level studies at all
- **D.** Grid resolution has no effect on accuracy

<details>
<summary>Show answer</summary>

**Answer: B** — This is the same nearest-cell approximation used throughout the workshop — it's a practical necessity of working with gridded satellite/model data, with an associated resolution trade-off.

</details>



In [ ]:
# Tests hidden — run your solution above first
pass

---
## 🏁 Recap

- Satellite/reanalysis products give us daily (or near-daily) environmental data at regular grid spacing — free via Google Earth Engine.
- NDVI from MODIS is a 16-day composite in reality; simplify carefully when teaching.
- The seasonal cycle in Zimbabwe shows temperature peaking in the dry season (around August–September) — exactly when the school year and many community health events occur.
- We now have (a) where each participant was every day, and (b) what the environment was like everywhere every day.

**Next:** Notebook 06 brings (a) and (b) together, computes trimester summaries, explores the data, and introduces the GeoAI interface.